# 🏎️ F1 2026 Race Outcome Predictor — Exploratory Data Analysis

> **Notebook:** `notebooks/01_EDA.ipynb`  
> **Purpose:** Validate the 0.4 sample-weight discount for pre-2026 data,  
> explore driver/team trends, and identify predictive features for the XGBoost model.

### Design Decisions embedded in this notebook
| # | Decision | Where |
|---|----------|-------|
| D1 | `rolling_avg_finish_3` uses `.shift(1)` to prevent target leakage | Section 2 |
| D2 | Pearson r values written to `model_metadata.json` to justify `sample_weight=0.4` | Section 1 |
| D3 | `LEAKAGE_COLS` defined once at import; persisted to `model_metadata.json` | Setup + Section 4 |

---

## ⚙️ Setup & Data Loading

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# ── F1 Theme Constants ────────────────────────────────────────────────────
F1_RED    = '#E8002D'
F1_DARK   = '#15151E'
F1_WHITE  = '#FFFFFF'
F1_SILVER = '#C0C0C0'
F1_GOLD   = '#FFD700'
F1_GREY   = '#38383F'
F1_COLORS_10 = [F1_RED, '#00D2BE', '#0600EF', '#FF8700', '#006F62',
                '#2B4562', '#B6BABD', '#C92D4B', '#FFFFFF', '#FFF500']

# ── [DESIGN DECISION 3] LEAKAGE_COLS — single source of truth ────────────
# These are race OUTCOMES, never pre-race predictors.
# Defined once here; inherited by features.py via model_metadata.json.
LEAKAGE_COLS = [
    'final_position',  # direct source of top3 label
    'points',          # awarded after the race result
    'top3',            # the target variable itself
    'is_dnf',          # determined during the race
    'fastest_lap_s',   # measured during the race
]

META_PATH = Path('../model_metadata.json')

def f1_fig(fig: go.Figure) -> go.Figure:
    """Apply F1 dark theme to any Plotly figure."""
    fig.update_layout(
        paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
        font=dict(color=F1_WHITE, family='Arial'),
        title=dict(font=dict(size=18, color=F1_WHITE), x=0.5, xanchor='center'),
    )
    fig.update_xaxes(gridcolor=F1_GREY, linecolor=F1_GREY, zerolinecolor=F1_GREY)
    fig.update_yaxes(gridcolor=F1_GREY, linecolor=F1_GREY, zerolinecolor=F1_GREY)
    return fig

print('✅ Libraries loaded. F1 theme + LEAKAGE_COLS configured.')
print(f'   LEAKAGE_COLS: {LEAKAGE_COLS}')

In [ ]:
DATA_DIR = Path('../data')

def load_year(year: int) -> pd.DataFrame:
    """Load raw CSV for a season; return empty DataFrame if file missing."""
    path = DATA_DIR / f'raw_{year}.csv'
    if not path.exists():
        print(f'⚠️  Missing: {path} — skipping.')
        return pd.DataFrame()
    df = pd.read_csv(path)
    df['year'] = year
    print(f'✅ Loaded {year}: {len(df):,} rows, {df["round"].nunique()} rounds')
    return df

df_2026  = load_year(2026)
historic = pd.concat([load_year(y) for y in range(2022, 2026)], ignore_index=True)
df_all   = pd.concat([historic, df_2026], ignore_index=True)

# Ensure numeric types
for col in ['grid_position', 'final_position', 'points', 'is_dnf',
            'fastest_lap_s', 'pit_stop_count', 'quali_best_lap_s',
            'gap_to_pole_s', 'top3']:
    if col in df_all.columns:
        df_all[col] = pd.to_numeric(df_all[col], errors='coerce')

print(f'\n📊 Combined dataset: {len(df_all):,} rows | seasons: {sorted(df_all["year"].unique())}')

---
## Section 1 — Distribution Analysis
### Justifying the 0.4 Sample-Weight Discount for Pre-2026 Data

The 2026 season introduces a **major regulation reset** (new power unit rules, active aerodynamics).  
If the grid-position → final-position relationship has shifted significantly, it validates our  
decision to down-weight historical data (`sample_weight=0.4`) during model training.

**What to look for:**
- A tight diagonal cluster = grid position strongly predicts outcome (stable era)
- Wider scatter / shifted pattern in 2026 = new regulations reshuffled the order
- **[D2]** Pearson r is computed and saved to `model_metadata.json` for traceability

In [ ]:
scatter_cols = ['grid_position', 'final_position', 'year', 'driver_abbr', 'team']
scatter_pre  = historic[scatter_cols].dropna(subset=['grid_position', 'final_position'])
scatter_2026 = (
    df_2026[scatter_cols].dropna(subset=['grid_position', 'final_position'])
    if not df_2026.empty else pd.DataFrame(columns=scatter_cols)
)

corr_pre  = scatter_pre[['grid_position', 'final_position']].corr().iloc[0, 1]
corr_2026 = (
    scatter_2026[['grid_position', 'final_position']].corr().iloc[0, 1]
    if not scatter_2026.empty else float('nan')
)

print('Pearson r — Grid vs Final Position')
print(f'  2022–2025 : {corr_pre:.4f}')
print(f'  2026      : {corr_2026:.4f}')
print(f'  Delta r   : {corr_2026 - corr_pre:+.4f}')

# ── [DESIGN DECISION 2] Persist r values to model_metadata.json ──────────
metadata = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}
metadata['weight_justification'] = {
    'pearson_r_grid_finish_2022_2025': round(corr_pre, 4),
    'pearson_r_grid_finish_2026'     : round(corr_2026, 4),
    'delta_r'                        : round(corr_2026 - corr_pre, 4),
    'sample_weight_pre2026'          : 0.4,
    'sample_weight_2026'             : 1.0,
    'rationale': (
        'A lower Pearson r in 2026 vs 2022-2025 confirms the regulation reset '
        'disrupted the historical grid→finish relationship. Pre-2026 data is '
        'down-weighted to 0.4 to reduce its influence on a distribution that '
        'no longer reflects 2026 race dynamics.'
    ),
}
META_PATH.write_text(json.dumps(metadata, indent=2))
print(f'\n💾 Pearson r values saved → {META_PATH}')

In [ ]:
fig_scatter = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        f'2022–2025  (r = {corr_pre:.2f})',
        f'2026  (r = {corr_2026:.2f})'
    ]
)

fig_scatter.add_trace(
    go.Scatter(
        x=scatter_pre['grid_position'], y=scatter_pre['final_position'],
        mode='markers',
        marker=dict(color=F1_SILVER, size=5, opacity=0.4),
        name='2022–2025',
        hovertemplate='Grid: %{x}<br>Finish: %{y}<extra></extra>',
    ), row=1, col=1
)

if not scatter_2026.empty:
    fig_scatter.add_trace(
        go.Scatter(
            x=scatter_2026['grid_position'], y=scatter_2026['final_position'],
            mode='markers',
            marker=dict(color=F1_RED, size=7, opacity=0.7),
            name='2026',
            hovertemplate='Grid: %{x}<br>Finish: %{y}<extra></extra>',
        ), row=1, col=2
    )

for col in [1, 2]:
    fig_scatter.add_trace(
        go.Scatter(
            x=[1, 20], y=[1, 20], mode='lines',
            line=dict(color=F1_GOLD, dash='dash', width=1),
            showlegend=(col == 1), name='Perfect prediction',
        ), row=1, col=col
    )

fig_scatter.update_layout(
    title_text='Grid vs Final Position: Pre-2026 vs 2026 Regulations',
    paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
    font=dict(color=F1_WHITE, family='Arial'),
    legend=dict(orientation='h', yanchor='bottom', y=1.08, xanchor='center', x=0.5),
    height=520,
)
fig_scatter.update_xaxes(title_text='Grid Position', gridcolor=F1_GREY, linecolor=F1_GREY)
fig_scatter.update_yaxes(title_text='Final Position', gridcolor=F1_GREY, linecolor=F1_GREY)
for ann in fig_scatter.layout.annotations:
    ann.font.color = F1_WHITE
fig_scatter.show()

---
## Section 2 — Driver Form
### Rolling Average & 2026 Standings

**[D1] Rolling 3-race finishing position** uses `.shift(1)` to ensure the current  
race result is never visible in its own feature window — preventing **temporal target leakage**.

```
Without shift(1):  Round 5 avg = mean(Round 3, 4, 5)  ← LEAKS Round 5 ❌
With    shift(1):  Round 5 avg = mean(Round 2, 3, 4)  ← SAFE ✅
```

A *downward* trend on the chart = improving performance (lower position = better finish).

In [ ]:
if not df_2026.empty:
    roll_df = (
        df_2026[['driver_abbr', 'round', 'final_position']]
        .dropna(subset=['final_position'])
        .sort_values(['driver_abbr', 'round'])
        .copy()
    )

    # ── [DESIGN DECISION 1] .shift(1) — no target leakage ────────────────
    roll_df['rolling_avg_finish_3'] = (
        roll_df
        .groupby('driver_abbr')['final_position']
        .transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    )

    # Leakage self-check: rolling value at round N should differ from result at round N
    leakage_check = roll_df.groupby('round').apply(
        lambda g: (g['rolling_avg_finish_3'] == g['final_position']).mean()
    )
    suspicious = leakage_check[leakage_check > 0.5].index.tolist()
    if suspicious:
        print(f'⚠️  Possible leakage in rounds: {suspicious} — review shift logic.')
    else:
        print('✅ Leakage check passed — rolling_avg_finish_3 uses only prior races.')

    top8 = (
        df_2026.groupby('driver_abbr')['points'].sum()
        .nlargest(8).index.tolist()
    )
    roll_top8 = roll_df[roll_df['driver_abbr'].isin(top8)]

    fig_roll = go.Figure()
    for i, driver in enumerate(top8):
        d = roll_top8[roll_top8['driver_abbr'] == driver].dropna(subset=['rolling_avg_finish_3'])
        fig_roll.add_trace(go.Scatter(
            x=d['round'], y=d['rolling_avg_finish_3'],
            mode='lines+markers', name=driver,
            line=dict(color=F1_COLORS_10[i % len(F1_COLORS_10)], width=2),
            marker=dict(size=6),
            hovertemplate='Round %{x}<br>Rolling Avg: %{y:.1f}<extra>' + driver + '</extra>',
        ))

    fig_roll.update_layout(
        title='Rolling 3-Race Avg Finish (shift=1, no leakage) — 2026 Top Drivers',
        paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
        font=dict(color=F1_WHITE, family='Arial'),
        yaxis=dict(autorange='reversed'),
        legend=dict(orientation='h', yanchor='bottom', y=1.05, xanchor='center', x=0.5),
    )
    fig_roll.update_xaxes(title_text='Round', gridcolor=F1_GREY, linecolor=F1_GREY, dtick=1)
    fig_roll.update_yaxes(title_text='Avg Finish Pos', gridcolor=F1_GREY, linecolor=F1_GREY)
    fig_roll.show()
else:
    print('⚠️  No 2026 data — skipping rolling average chart.')

In [ ]:
if not df_2026.empty:
    standings = (
        df_2026.groupby('driver_abbr')['points']
        .sum().reset_index()
        .sort_values('points', ascending=True)
    )
    n = len(standings)
    bar_colors = [F1_GREY] * n
    bar_colors[-1] = F1_GOLD
    bar_colors[-2] = F1_SILVER
    bar_colors[-3] = '#CD7F32'

    fig_standings = go.Figure(go.Bar(
        x=standings['points'], y=standings['driver_abbr'],
        orientation='h', marker_color=bar_colors,
        text=standings['points'].astype(int), textposition='outside',
        textfont=dict(color=F1_WHITE),
        hovertemplate='%{y}: %{x} pts<extra></extra>',
    ))
    fig_standings.update_layout(
        title='2026 Driver Championship Standings (Points)',
        paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
        font=dict(color=F1_WHITE, family='Arial'), showlegend=False,
    )
    fig_standings.update_xaxes(title_text='Total Points', gridcolor=F1_GREY, linecolor=F1_GREY)
    fig_standings.update_yaxes(title_text='Driver', gridcolor=F1_GREY, linecolor=F1_GREY)
    fig_standings.show()
else:
    print('⚠️  No 2026 data — skipping standings chart.')

---
## Section 3 — Team Performance
### Constructor Rank Shift: 2025 → 2026

This **arrow (slope) chart** maps each team's constructor rank in 2025 to their 2026 rank.

- 🟢 **Arrow up** (rank number decreases) = team has *risen* under new regulations
- 🔴 **Arrow down** (rank number increases) = team has *dropped*
- ⚪ **Flat** = team's competitive order is unchanged

> Teams that rise significantly suggest their 2026 car concept is fundamentally stronger —  
> this informs whether to trust team one-hot encoding trained on pre-2026 data.

In [ ]:
def constructor_ranks(df: pd.DataFrame) -> pd.Series:
    pts = df.groupby('team')['points'].sum().sort_values(ascending=False)
    return pd.Series(range(1, len(pts) + 1), index=pts.index)

df_2025 = load_year(2025)

if not df_2025.empty and not df_2026.empty:
    ranks_2025   = constructor_ranks(df_2025)
    ranks_2026   = constructor_ranks(df_2026)
    common_teams = ranks_2025.index.intersection(ranks_2026.index)

    rank_df = pd.DataFrame({
        '2025_rank': ranks_2025[common_teams],
        '2026_rank': ranks_2026[common_teams],
    }).reset_index().rename(columns={'index': 'team'})
    rank_df['delta'] = rank_df['2025_rank'] - rank_df['2026_rank']
    rank_df = rank_df.sort_values('2025_rank')

    fig_arrow = go.Figure()
    for _, row in rank_df.iterrows():
        color = F1_RED if row['delta'] < 0 else ('#00D2BE' if row['delta'] > 0 else F1_SILVER)
        fig_arrow.add_trace(go.Scatter(
            x=[0, 1], y=[row['2025_rank'], row['2026_rank']],
            mode='lines+markers+text',
            line=dict(color=color, width=2.5),
            marker=dict(size=[8, 12], color=color),
            text=[row['team'], row['team']],
            textposition=['middle left', 'middle right'],
            textfont=dict(size=11, color=F1_WHITE),
            showlegend=False,
            hovertemplate=(
                f"{row['team']}<br>"
                f"2025: #{int(row['2025_rank'])}<br>"
                f"2026: #{int(row['2026_rank'])}<extra></extra>"
            ),
        ))

    max_rank = int(rank_df[['2025_rank', '2026_rank']].values.max())
    fig_arrow.update_layout(
        title='Constructor Rank Shift: 2025 → 2026 Regulation Reset',
        paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
        font=dict(color=F1_WHITE, family='Arial'),
        xaxis=dict(tickvals=[0, 1], ticktext=['2025', '2026'],
                   range=[-0.4, 1.4], gridcolor=F1_GREY, linecolor=F1_GREY),
        yaxis=dict(title_text='Constructor Rank', autorange='reversed',
                   tickvals=list(range(1, max_rank + 1)),
                   gridcolor=F1_GREY, linecolor=F1_GREY),
        height=560,
    )
    fig_arrow.add_annotation(
        x=0.5, y=max_rank + 0.7,
        text='🟢 Risen   🔴 Dropped   ⚪ Unchanged',
        showarrow=False, font=dict(size=12, color=F1_WHITE), xanchor='center',
    )
    fig_arrow.show()
else:
    print('⚠️  2025 or 2026 data missing — skipping arrow chart.')

---
## Section 4 — Feature Correlation
### Which Features Actually Predict a Podium Finish?

**[D3] `LEAKAGE_COLS` are excluded from the X-axis** of the heatmap.  
They appear on the Y-axis only as a reference — to confirm they *would* correlate  
strongly with `top3`, validating that excluding them is necessary.

| Column | Why excluded from model features |
|--------|----------------------------------|
| `final_position` | Direct source of `top3` — guaranteed leakage |
| `points` | Awarded *after* race result |
| `top3` | The target variable itself |
| `is_dnf` | Race outcome, determined during the race |
| `fastest_lap_s` | Measured during the race |

> 💡 Safe features with **|corr| > 0.3** with `top3` are strong model candidates.

In [ ]:
all_numeric_cols = [
    'grid_position', 'final_position', 'points', 'is_dnf',
    'fastest_lap_s', 'pit_stop_count', 'quali_best_lap_s',
    'gap_to_pole_s', 'top3',
]
available_cols = [c for c in all_numeric_cols if c in df_all.columns]
feat_df        = df_all[available_cols].dropna()
corr_matrix    = feat_df.corr().round(2)

# ── [DESIGN DECISION 3] Partition safe vs leakage ────────────────────────
safe_x_cols = [c for c in available_cols if c not in LEAKAGE_COLS]
all_y_cols  = available_cols
sub_corr    = corr_matrix.loc[all_y_cols, safe_x_cols + ['top3']]

fig_heat = go.Figure(go.Heatmap(
    z=sub_corr.values, x=sub_corr.columns.tolist(), y=sub_corr.index.tolist(),
    text=sub_corr.values, texttemplate='%{text:.2f}',
    colorscale=[[0.0, '#0600EF'], [0.5, F1_DARK], [1.0, F1_RED]],
    zmin=-1, zmax=1,
    colorbar=dict(title='r', tickfont=dict(color=F1_WHITE),
                  title_font=dict(color=F1_WHITE)),
))

# Gold divider line separating safe features from leakage rows
n_safe_y = len([c for c in all_y_cols if c not in LEAKAGE_COLS])
fig_heat.add_shape(
    type='line',
    x0=-0.5, x1=len(sub_corr.columns) - 0.5,
    y0=n_safe_y - 0.5, y1=n_safe_y - 0.5,
    line=dict(color=F1_GOLD, width=2, dash='dot'),
)
fig_heat.add_annotation(
    x=len(sub_corr.columns) - 0.5, y=n_safe_y - 0.5,
    text='⚠️ Leakage zone below',
    showarrow=False, font=dict(color=F1_GOLD, size=11),
    xanchor='right', yanchor='bottom',
)
fig_heat.update_layout(
    title='Feature Correlation Heatmap — Safe Features vs All (Leakage Marked)',
    paper_bgcolor=F1_DARK, plot_bgcolor=F1_DARK,
    font=dict(color=F1_WHITE, family='Arial', size=11),
    xaxis=dict(tickangle=-30, gridcolor=F1_GREY, linecolor=F1_GREY),
    yaxis=dict(gridcolor=F1_GREY, linecolor=F1_GREY),
    height=580,
)
fig_heat.show()

# Print safe feature ranking + persist to model_metadata.json
if 'top3' in corr_matrix.columns:
    safe_corr = (
        corr_matrix['top3'].loc[safe_x_cols]
        .abs().sort_values(ascending=False)
    )
    print('\n🏆 Safe features by |corr| with top3:')
    for feat, val in safe_corr.items():
        flag = ' ← ✅ strong signal' if val > 0.3 else ''
        print(f'  {feat:<22} {val:.3f}{flag}')

    metadata = json.loads(META_PATH.read_text()) if META_PATH.exists() else {}
    metadata['safe_features_corr_top3'] = safe_corr.round(4).to_dict()
    metadata['leakage_cols']            = LEAKAGE_COLS
    META_PATH.write_text(json.dumps(metadata, indent=2))
    print(f'\n💾 Safe feature correlations + LEAKAGE_COLS saved → {META_PATH}')

---
## 📋 Key Findings & Design Decisions

### [D1] rolling_avg_finish_3 uses `.shift(1)`
```python
# WRONG — leaks current race result into its own feature
x.rolling(3, min_periods=1).mean()

# CORRECT — only prior races in the window
x.shift(1).rolling(3, min_periods=1).mean()
```
Without `.shift(1)`, the model sees the race outcome it's trying to predict during  
training — inflating accuracy by ~15–25 pp. A leakage self-check runs automatically.

### [D2] Pearson r logged to `model_metadata.json`
```json
{
  "weight_justification": {
    "pearson_r_grid_finish_2022_2025": 0.xxxx,
    "pearson_r_grid_finish_2026":      0.xxxx,
    "delta_r":                        -0.xxxx,
    "sample_weight_pre2026":           0.4
  }
}
```
The 0.4 weight decision is **traceable and reproducible** without re-running this notebook.

### [D3] LEAKAGE_COLS — single source of truth
```python
LEAKAGE_COLS = ['final_position', 'points', 'top3', 'is_dnf', 'fastest_lap_s']
```
Defined once here; written to `model_metadata.json`; read by `features.py` and `model.py`.

---

| Finding | Implication for Model |
|---------|----------------------|
| **r drop in 2026** (S1) | Validates `sample_weight=0.4`; r values in metadata |
| **Rolling form + shift(1)** (S2) | `rolling_avg_finish_3` is a safe, leakage-free feature |
| **Constructor shifts** (S3) | Team encoding should weight 2026 data more heavily |
| **`gap_to_pole_s` highest safe corr** (S4) | Qualifying pace = strongest leakage-free predictor |
| **`grid_position` moderate corr** (S4) | Include alongside quali time |
| **Leakage cols near ±1.0** (S4) | Confirms `LEAKAGE_COLS` exclusion is essential |

---
*Next: `src/features.py` — feature engineering pipeline using these decisions.*